# Part 3 — Deploy your own JupyterHub with Helm

In Part 2 you *used* NRP's services. Now you'll *run your own*: deploy a custom
**JupyterHub** on NRP — the same kind of multi-user hub you're sitting in right
now — into a namespace reserved for you, using **Helm**.

**What you'll do**

1. Verify you can deploy into your assigned namespace.
2. Add the JupyterHub Helm chart and configure it with a values file.
3. Deploy the hub, expose it at a URL, and customize images / resources / storage.
4. Manage and tear it down.
5. **Bonus:** reuse Part 2's RAG pattern to ask the JupyterHub docs how to
   customize your hub.

> **How to run this part.** The `helm` and `kubectl` commands below are meant for
> a **terminal** (open one from the JupyterLab launcher). They're shown as
> copy-paste blocks. A couple of cells in this notebook are *runnable* checks —
> those are grey code cells you can Shift+Enter.

> 📘 **Docs:** [Deploy JupyterHub on NRP](https://nrp.ai/documentation/userdocs/jupyter/jupyterhub/) ·
> [JupyterHub service](https://nrp.ai/documentation/userdocs/jupyter/jupyterhub-service/) ·
> [Zero-to-JupyterHub (upstream)](https://z2jh.jupyter.org)

## Key concepts

- **Helm** is a package manager for Kubernetes: instead of writing every
  Deployment, Service, and ConfigMap by hand, you install a **chart** (a bundle
  of templates) and tune it through a **values file**.
- The **JupyterHub Helm chart** deploys a hub, a proxy, and per-user notebook
  servers. You customize authentication, images, CPU/RAM/GPU limits, and storage
  in the values file.
- **One hub per namespace.** A JupyterHub can only be deployed once per
  namespace — that's why each of you has your **own** namespace
  (`nrp-training-000` … `nrp-training-099`, on the slip you were handed).
- In the **training JupyterHub** terminal, `kubectl`, `helm`, and your
  kubeconfig are already wired up — no local install needed.

## 1. Set your namespace and verify access

Put **your** assigned namespace below and run the cell. It should print `yes`.

In [ ]:
import subprocess

NAMESPACE = "nrp-training-000"   # <-- CHANGE this to YOUR assigned namespace (from your slip)

ans = subprocess.run(
    ["kubectl", "auth", "can-i", "create", "deployment", "-n", NAMESPACE],
    capture_output=True, text=True,
).stdout.strip()
print(f"Can I deploy into {NAMESPACE}?  ->  {ans}")
print("Expected: yes.  If you see 'no', double-check the namespace name on your slip.")

## 2. Add the JupyterHub Helm repository

Run in a **terminal** (replace `<namespace>` with yours):

```bash
helm repo add jupyterhub https://jupyterhub.github.io/helm-chart/ -n <namespace>
helm repo update
helm repo list
```

Confirm `helm` and `kubectl` are wired up first:

```bash
kubectl auth whoami && helm version --short
```

## 3. Configure: the values file

This repo ships a complete, **NRP-tested** values file at
[`yamls/jhub-values.yaml`](yamls/jhub-values.yaml) — it deploys cleanly as your
workshop service account. Open it and skim the comments; the parts you'll most
likely change are the **password** (`hub.config.DummyAuthenticator.password`) and
the **image profiles** (`singleuser.profileList`).

A few settings in it are non-obvious but **required on NRP** (the chart's
defaults would be rejected):

- `hub.service.type` / `proxy.service.type: ClusterIP` — NRP blocks
  `LoadBalancer` services.
- `hub.resources` and `proxy.chp.resources` with **both** requests *and* limits —
  NRP requires every container to declare them.
- `scheduling.userScheduler` / `userPlaceholder` and `prePuller` **disabled** —
  those need *cluster-scoped* RBAC your namespace service account doesn't have.
- `storageClass: rook-ceph-block-east` + node affinity `us-east` — matches where
  the training cluster and its storage live.

> We ship this file precisely because the stock chart defaults fail on NRP for
> the reasons above.

## 4. Deploy with Helm

From the repo directory (`cd ~/cra-tutorial`), pick a release name (e.g. `jhub`)
and deploy into **your** namespace with the shipped values file:

```bash
helm upgrade --cleanup-on-fail --install jhub jupyterhub/jupyterhub \\
  --namespace <namespace> \\
  --version 3.3.7 \\
  --values yamls/jhub-values.yaml \\
  --wait --timeout 10m
```

The proxy secret is auto-generated by the chart — nothing to paste. After
~1–2 minutes you'll see `STATUS: deployed`. Watch the pods come up:

```bash
kubectl get pods -n <namespace>
```

You should see a **hub** pod and a **proxy** pod reach `Running` (this exact flow
was tested end-to-end as the workshop service account). User pods are created on
demand when someone logs in.

You can also watch your namespace from here (re-run after deploying):

In [ ]:
import subprocess
print(subprocess.run(["kubectl","get","pods,svc","-n",NAMESPACE],
                     capture_output=True, text=True).stdout or "(nothing deployed yet)")

## 5. Make it reachable (Ingress)

By default the hub is internal (`ClusterIP`). Expose it at a public URL by adding
an `ingress` block to your values file, then re-running the `helm upgrade`:

```yaml
ingress:
  enabled: true
  ingressClassName: haproxy
  hosts: ["<your-hub-name>.nrp-nautilus.io"]
  tls:
    - hosts: ["<your-hub-name>.nrp-nautilus.io"]
```

Pick a unique `<your-hub-name>`. After redeploying, browse to
`https://<your-hub-name>.nrp-nautilus.io` and log in with `admin` /
`training123`.

## 6. Customize

The power of the values file is that re-running `helm upgrade` reconfigures the
hub. A few common knobs:

**Let users pick an image / size** — add a `profileList`:

```yaml
singleuser:
  profileList:
  - display_name: "Small (2 CPU, 4 GB)"
    default: true
    kubespawner_override: {cpu_limit: 2, cpu_guarantee: 2, mem_limit: 4G, mem_guarantee: 4G}
  - display_name: "Large (8 CPU, 16 GB)"
    kubespawner_override: {cpu_limit: 8, cpu_guarantee: 8, mem_limit: 16G, mem_guarantee: 16G}
  - display_name: "PyTorch + GPU"
    kubespawner_override:
      image_spec: quay.io/jupyter/pytorch-notebook:cuda12-2024-04-22
      extra_resource_limits: {"nvidia.com/gpu": "1"}
```

**Shared read-only data** for all users — mount an existing PVC:

```yaml
singleuser:
  storage:
    extraVolumes:      [{name: shared, persistentVolumeClaim: {claimName: jupyterhub-shared-volume}}]
    extraVolumeMounts: [{name: shared, mountPath: /home/shared}]
```

How do you find every available knob? **That's what the Bonus below is for** —
ask the docs.

## 7. Manage and clean up

```bash
helm list -n <namespace>                                  # your releases
kubectl logs -n <namespace> -l component=hub --tail=50    # hub logs
```

**Please uninstall when you're done** so the namespace is free:

```bash
helm uninstall jhub -n <namespace>
# user data PVCs are kept; to delete them too (careful!):
# kubectl delete pvc -n <namespace> -l component=singleuser-storage
```

## Bonus (optional): ask the *whole* JupyterHub docs with RAG

<details>
<summary><b>Click to expand — index the entire Zero-to-JupyterHub docs and ask it anything</b></summary>

Customizing a hub means digging through the **Zero-to-JupyterHub** docs
([z2jh.jupyter.org](https://z2jh.jupyter.org/en/stable/)). Instead, point the
*exact RAG pattern from Part 2* at the **whole documentation set** — every page —
and just ask.

The first cell below builds the index once (~30 s). After that, call
`ask_docs("your question")` as many times as you like — it answers **only** from
the docs and cites which page it used. Same technique as Part 2's NRP-docs RAG,
scaled from one page to the entire site, and genuinely handy while you tune your
`jhub-values.yaml`.

</details>

In [ ]:
# Build a RAG index over the ENTIRE Zero-to-JupyterHub docs (same pattern as Part 2).
# Runs once (~30 s); then call ask_docs("...") below. Self-contained.
import os, re, requests, numpy as np
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=os.environ["OPENAI_API_BASE"])
BASE = "https://raw.githubusercontent.com/jupyterhub/zero-to-jupyterhub-k8s/main/docs/source/"

# Every docs page in the z2jh repo (docs/source/**.md), minus the changelog.
PAGES = [
    "administrator/advanced", "administrator/architecture", "administrator/authentication",
    "administrator/cost", "administrator/debug", "administrator/index",
    "administrator/optimization", "administrator/security", "administrator/services",
    "administrator/troubleshooting", "administrator/upgrading/index",
    "administrator/upgrading/upgrade-1-to-2", "administrator/upgrading/upgrade-2-to-3",
    "administrator/upgrading/upgrade-3-to-4", "index", "jupyterhub/customization",
    "jupyterhub/customizing/extending-jupyterhub",
    "jupyterhub/customizing/user-environment", "jupyterhub/customizing/user-management",
    "jupyterhub/customizing/user-resources", "jupyterhub/customizing/user-storage",
    "jupyterhub/index", "jupyterhub/installation", "jupyterhub/uninstall",
    "kubernetes/amazon/efs_storage", "kubernetes/amazon/step-zero-aws-eks",
    "kubernetes/amazon/step-zero-aws", "kubernetes/digital-ocean/step-zero-digital-ocean",
    "kubernetes/docker-desktop/step-zero-docker-desktop",
    "kubernetes/google/step-zero-gcp", "kubernetes/ibm/step-zero-ibm", "kubernetes/index",
    "kubernetes/microsoft/step-zero-azure", "kubernetes/minikube/step-zero-minikube",
    "kubernetes/other-infrastructure/step-zero-other", "kubernetes/ovh/step-zero-ovh",
    "kubernetes/redhat/step-zero-openshift", "kubernetes/setup-helm",
    "kubernetes/setup-kubernetes", "repo2docker", "resources/community",
    "resources/glossary", "resources/index", "resources/reference-docs", "resources/tools"
]

def _clean(t):
    t = re.sub(r"```\{[^}]*\}", "", t)            # MyST directive openers
    t = re.sub(r"```.*?```", "", t, flags=re.S)     # fenced code blocks
    t = re.sub(r"\{ref\}`[^`]*`", "", t)           # cross-references
    t = re.sub(r"\{[a-z]+\}`([^`]*)`", r"\1", t)   # other roles -> keep the text
    return re.sub(r"\n{3,}", "\n\n", t).strip()

chunks, pages = [], []
for p in PAGES:
    try:
        txt = _clean(requests.get(BASE + p + ".md", timeout=20).text)
    except Exception:
        continue
    for i in range(0, len(txt), 550):
        ch = txt[i:i+700]
        if len(ch) > 120:
            chunks.append(ch); pages.append(p)

def embed(texts):
    vecs = []
    for i in range(0, len(texts), 64):              # batch so each request stays small
        r = client.embeddings.create(model="qwen3-embedding", input=texts[i:i+64])
        vecs += [d.embedding for d in r.data]
    v = np.array(vecs)
    return v / np.linalg.norm(v, axis=1, keepdims=True)

print(f"Indexing {len(chunks)} chunks from {len(set(pages))} JupyterHub doc pages...")
chunk_vecs = embed(chunks)

def ask_docs(question, k=4):
    """Ask the whole JupyterHub docs. Answers only from the docs, cites the page."""
    qv = embed([question])[0]
    top = (chunk_vecs @ qv).argsort()[-k:][::-1]
    context = "\n\n".join(f"[{pages[i]}]\n{chunks[i]}" for i in top)
    r = client.chat.completions.create(
        model="minimax-m2", max_tokens=1200, temperature=0.2,
        messages=[
            {"role": "system", "content": "Answer using ONLY the JupyterHub docs context. "
             "Be concise, show the relevant Helm values YAML when applicable, and cite the [page] you used."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ])
    m = r.choices[0].message
    return m.content or getattr(m, "reasoning", None) or "(no answer)"

print("Ready. Ask anything with  ask_docs(\"...\")")

In [ ]:
# Ask the docs anything — edit these or add your own questions.
for q in [
    "How do I give each user more memory and CPU?",
    "How do I let users choose a GPU image?",
    "How do I use GitHub authentication instead of a shared password?",
]:
    print("Q:", q)
    print(ask_docs(q))
    print("=" * 78)